In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/lakeflow_pipeline/default/data/orders/',
  format => 'json',
  inferSchema => 'true',
  multiline => 'true'
)
limit 2;

In [0]:
%sql
-- JSON -> Bronze Layer
-- Read all files from your working directory each time the query executed

CREATE OR REPLACE TABLE lakeflow_pipeline.`1_bronze_db`.tb_orders_bronze
AS 
SELECT 
  *,
  _metadata.file_path AS source_file_path,
  _metadata.file_modification_time AS source_file_modification_time,
  _metadata.file_name as source_file_name,
  current_timestamp() AS ingestion_ts
FROM read_files(
  '/Volumes/lakeflow_pipeline/default/data/orders/',
  format => 'json',
  inferSchema => 'true',
  multiline => 'true'
)
;

In [0]:
%sql
SELECT * FROM lakeflow_pipeline.`1_bronze_db`.tb_orders_bronze limit 2
;

In [0]:
%sql
-- Bronze => Silver
-- Read the entire bronze table each time the query executed

-- 1. Extract the required columns
-- 2. Rename the columns
-- 3. Add the ingestion_ts column
CREATE OR REPLACE TABLE lakeflow_pipeline.`2_silver_db`.tb_orders_silver
AS 
SELECT
  order_id,
  customer_id,
  category,
  channel,  
  discount_pct,
  final_status,  
  to_date(cast(order_ts as timestamp)) as order_date,	
  product_id,
  product_name, 
  quantity,
  unit_price,
  order_amount,
  current_timestamp() as ingestion_ts
FROM lakeflow_pipeline.`1_bronze_db`.tb_orders_bronze
;

-- 4. Display the silver table 
SELECT * FROM lakeflow_pipeline.`2_silver_db`.tb_orders_silver limit 2

In [0]:
%sql
-- Silver => Gold
-- Read the entire silver table each time the query executed

-- 1. Extract the required columns
-- 2. Rename the columns
-- 3. Add the ingestion_ts column
CREATE OR REPLACE VIEW lakeflow_pipeline.`3_gold_db`.vw_daily_orders_summary
AS 
SELECT
  order_date,	
  count(distinct customer_id) as total_customers,
  count(distinct order_id) as total_orders,
  sum(quantity) as total_quantity,
  sum(order_amount) as total_order_amount,
  (total_order_amount - sum(discount_pct*order_amount/100)) as total_net_order_amount,
  sum(case when final_status = 'completed' then 1 else 0 end) as total_completed_orders,
  sum(case when final_status = 'completed' then order_amount else 0 end) as total_completed_order_amount,
  sum(case when final_status = 'canceled' then 1 else 0 end) as total_canceled_orders,
  sum(case when final_status = 'canceled' then order_amount else 0 end) as total_canceled_order_amount,
  current_timestamp() as ingestion_ts
FROM lakeflow_pipeline.`2_silver_db`.tb_orders_silver
GROUP BY order_date
;

-- 4. Display the gold view 
SELECT * FROM lakeflow_pipeline.`3_gold_db`.vw_daily_orders_summary limit 2;